In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-pinecone \
    langchain-groq \
    pinecone \
    sentence-transformers \
    python-dotenv \
    pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

def load_pdf_file(data):
    loader = PyPDFLoader(data)
    documents = loader.load()
    return documents

In [ ]:
import os
print(os.path.exists("constitution.pdf"))

In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-pinecone \
    langchain-groq \
    pinecone \
    sentence-transformers \
    python-dotenv \
    pypdf

In [ ]:
extracted_data = load_pdf_file("constitution.pdf")

In [ ]:
#Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [ ]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
#Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [ ]:
embeddings=download_hugging_face_embeddings()

In [ ]:
query_result=embeddings.embed_query("This is my first project")
print("Length",len(query_result))

In [ ]:
import os

# This one code works for BOTH Colab and Render / GitHub
try:
    # For Colab
    from google.colab import userdata
    groq_api_key = userdata.get('GROQ_API_KEY')
except:
    # For Laptop / Render / HuggingFace
    groq_api_key = os.getenv("GROQ_API_KEY")

print("Key loaded:", "Yes" if groq_api_key else "No - Add it in Secrets!")

In [ ]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
GROQ_API_KEY=os.environ.get('GROQ_API_KEY')

In [ ]:
import os
from pinecone import Pinecone, ServerlessSpec
from google.colab import userdata

# --- SAFE WAY - No key visible ---
# First add keys in Colab Secrets (left side key icon)
# Name: PINECONE_API_KEY and GROQ_API_KEY
try:
    os.environ["PINECONE_API_KEY"] = userdata.get('PINECONE_API_KEY')
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
except Exception as e:
     print("error loading secrets")
    # Fallback for local testing - DO NOT put real key here


pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "constitution-chatbot"

# NEW WAY to check indexes
existing_indexes = [index.name for index in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created successfully.")
else:
    print("Index already exists. Using existing index.")

In [ ]:
# Load Existing index

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [ ]:
docsearch

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
retrieved_docs = retriever.invoke("What is right to freedom")

In [ ]:
retrieved_docs

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(
        model="openai/gpt-oss-20b",
        temperature=0
)

In [ ]:
# ChatPromptTemplate: Creates a structured prompt template for sending
# system instructions and user messages to the LLM.
from langchain_core.prompts import ChatPromptTemplate
# RunnablePassthrough: Passes the input data through a LangChain chain
# without changing it.
from langchain_core.runnables import RunnablePassthrough
# StrOutputParser: Converts the LLM's response into a simple string.
from langchain_core.output_parsers import StrOutputParser


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "Context:\n{context}"
)

# Creates a chat prompt containing a system message and a human/user message.
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
response = rag_chain.invoke("If the Government imposes greater control and censorship over social media platforms to curb fake news, which fundamental rights under Article 19(1)(a), Article 19(2), and Article 21 (Right to Privacy from Puttaswamy judgment) will come into conflict, and how did the Supreme Court in Shreya Singhal v. Union of India resolve this balance?")
print(response)

In [ ]:
response=rag_chain.invoke("explain any 1 fundamental right")
print(response)





In [ ]:
!pip install -q langchain langchain-huggingface langchain-pinecone langchain-groq gradio pinecone

In [ ]:
import gradio as gr
import os

# Your rag_chain must be created in SAME cell or above
# Example: make sure this part is there before chatbot function
# If you already created rag_chain in previous cell, copy that code here

def samvidhan_chatbot(question):
    if not question or not question.strip():
        return "Please ask a question"
    try:
        # rag_chain must exist
        answer = rag_chain.invoke(question)
        return answer
    except Exception as e:
        return f"Error: {e}"

demo = gr.Interface(
    fn=samvidhan_chatbot,
    inputs=gr.Textbox(label="Ask about Constitution", placeholder="What is Article 21?"),
    outputs=gr.Textbox(label="Answer", lines=8),
    title="Samvidhan AI",
    description="Indian Constitution Assistant"
)

demo.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=True  # <-- This gives you the public link
)

In [ ]:
%%writefile requirements.txt
gradio
langchain
langchain-huggingface
langchain-pinecone
langchain-groq
pinecone
sentence-transformers
groq

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/Samvidhan_AI"
!cp /content/app.py "/content/drive/MyDrive/Samvidhan_AI/"
!cp /content/requirements.txt "/content/drive/MyDrive/Samvidhan_AI/"